- [ ] What predicts overtakes made  starting position, pace delta vs. the car ahead, tyre delta? (multiple regression)
- [ ] Are the biggest position swings mostly explained by pit cycles and Safety Cars (as suspected from the Norris case), or is there a residual, unexplained portion once those are controlled for? (regression with pit-stop/Safety-Car dummy variables, examine residuals)

- [ ] Does spending more time "fighting" (within the 1.0s DRS zone) correlate with more overtakes attempted or made? (correlation)
- [ ] Does being lapped correlate more with reliability issues (damage, mechanical) or a pure pace deficit? (compare group differences lapped vs. not, by cause)

In [1]:
import sys
sys.path.append("../")
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm


import sqlite3

conn = sqlite3.connect("../DATA INGESTION/f1.db")

- [X] What predicts overtakes made  starting position, pace delta vs. the car ahead, tyre delta? (multiple regression)

In [8]:
overtakes_query = """
SELECT ot.session_key, ot.overtaking_driver_number AS driver_number,
       COUNT(*) AS overtakes_made
FROM silver_overtakes ot
JOIN silver_sessions s ON ot.session_key = s.session_key AND s.session_name = 'Race'
GROUP BY ot.session_key, ot.overtaking_driver_number
"""
df_overtakes = pd.read_sql(overtakes_query, conn)
df_overtakes.shape
df_overtakes['overtakes_made'].describe()

count    1435.000000
mean       12.680836
std         7.918422
min         1.000000
25%         7.000000
50%        12.000000
75%        17.000000
max       101.000000
Name: overtakes_made, dtype: float64

In [12]:
grid_query = """
SELECT g.driver_number, g.position AS grid_position, gs.meeting_key,
       (SELECT rs.session_key FROM silver_sessions rs 
        WHERE rs.meeting_key = gs.meeting_key AND rs.session_name = 'Race' LIMIT 1) AS session_key
FROM silver_starting_grid g
JOIN silver_sessions gs ON g.session_key = gs.session_key
WHERE gs.session_name = 'Qualifying'
"""
df_grid = pd.read_sql(grid_query, conn)
df_grid.shape


(1430, 4)

In [14]:
pace_query = """
SELECT l.session_key, l.driver_number,
       AVG(l.lap_duration) AS mean_race_pace,
       COUNT(*) AS n_laps
FROM silver_laps l
JOIN silver_sessions s ON l.session_key = s.session_key AND s.session_name = 'Race'
WHERE l.lap_duration IS NOT NULL
  AND l.lap_duration BETWEEN 60 AND 200
  AND l.is_pit_out_lap = 0
GROUP BY l.session_key, l.driver_number
"""
df_pace = pd.read_sql(pace_query, conn)
df_pace.shape
#df_pace['n_laps'].describe()


df_pace = df_pace[df_pace['n_laps'] >= 25].copy()
df_pace.shape

(1366, 4)

In [15]:
pit_count_query = """
SELECT p.session_key, p.driver_number, COUNT(*) AS pit_count
FROM silver_pit p
JOIN silver_sessions s ON p.session_key = s.session_key AND s.session_name = 'Race'
GROUP BY p.session_key, p.driver_number
"""
df_pit_count = pd.read_sql(pit_count_query, conn)
df_pit_count.shape
df_pit_count['pit_count'].value_counts().sort_index()

pit_count
1    511
2    556
3    111
4     46
5     39
6     11
7      4
Name: count, dtype: int64

In [19]:
results_query = """
SELECT sr.session_key, sr.driver_number, sr.position AS final_position
FROM silver_session_result sr
JOIN silver_sessions s ON sr.session_key = s.session_key AND s.session_name = 'Race'
WHERE sr.dnf = 0 AND sr.dns = 0 AND sr.dsq = 0
  AND sr.position IS NOT NULL
"""
df_results = pd.read_sql(results_query, conn)
df_results.shape
df_results.head()

,session_key,driver_number,final_position
0,7779,11,1
1,7779,1,2
2,7779,14,3
3,7779,63,4
4,7779,44,5


In [42]:
df_master = df_results.merge(df_overtakes, on=['session_key','driver_number'], how='left')
df_master = df_master.merge(df_grid, on=['session_key','driver_number'], how='left')
df_master = df_master.merge(df_pace[['session_key','driver_number','mean_race_pace']], 
                             on=['session_key','driver_number'], how='left')
df_master = df_master.merge(df_pit_count, on=['session_key','driver_number'], how='left')

df_master.shape
df_master.isna().sum()
df_master.head()

,session_key,driver_number,final_position,overtakes_made,grid_position,meeting_key,mean_race_pace,pit_count
0,7779,11,1,1.0,1.0,1142.0,95.610449,NaN
1,7779,1,2,13.0,15.0,1142.0,96.188796,NaN
2,7779,14,3,1.0,2.0,1142.0,96.159857,NaN
3,7779,63,4,NaN,3.0,1142.0,96.441531,NaN
4,7779,44,5,5.0,7.0,1142.0,96.854429,NaN


In [43]:
df_master['overtakes_made'] = df_master['overtakes_made'].fillna(0)
df_master = df_master.dropna(subset=['grid_position','mean_race_pace','pit_count']).copy()
df_master.shape

(1024, 8)

In [48]:
if 'pace_of_car_ahead' in df_master.columns:
    df_master = df_master.drop(columns=['pace_of_car_ahead','pace_delta'], errors='ignore')

df_ahead = df_master[['session_key','final_position','mean_race_pace']].copy()
df_ahead['final_position'] = df_ahead['final_position'] + 1
df_ahead = df_ahead.rename(columns={'mean_race_pace':'pace_of_car_ahead'})

df_master = df_master.merge(df_ahead, on=['session_key','final_position'], how='left')
df_master['pace_delta'] = df_master['mean_race_pace'] - df_master['pace_of_car_ahead']

df_master[['pace_delta']].describe()


,pace_delta
count,963.000000
mean,0.126938
std,0.480081
min,-2.851970
25%,-0.015733
50%,0.087695
75%,0.314632
max,2.919902


In [49]:
df_master = df_master.dropna(subset=['pace_delta']).copy()
df_master.shape
df_master[['overtakes_made','grid_position','pace_delta','pit_count']].describe()

,overtakes_made,grid_position,pace_delta,pit_count
count,963.000000,963.000000,963.000000,963.000000
mean,12.913811,10.766355,0.126938,1.900312
std,8.701677,5.594996,0.480081,1.080170
min,0.000000,1.000000,-2.851970,1.000000
25%,7.000000,6.000000,-0.015733,1.000000
50%,12.000000,11.000000,0.087695,2.000000
75%,18.000000,16.000000,0.314632,2.000000
max,101.000000,22.000000,2.919902,7.000000


In [50]:
import statsmodels.formula.api as smf

model = smf.ols(
    'overtakes_made ~ grid_position + pace_delta + pit_count',
    data=df_master
).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         overtakes_made   R-squared:                       0.114
Model:                            OLS   Adj. R-squared:                  0.111
Method:                 Least Squares   F-statistic:                     41.22
Date:                Mon, 20 Jul 2026   Prob (F-statistic):           4.67e-25
Time:                        10:06:41   Log-Likelihood:                -3391.0
No. Observations:                 963   AIC:                             6790.
Df Residuals:                     959   BIC:                             6809.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         5.4871      0.738      7.439


FIRST ATTEMPT — RACE-LEVEL REGRESSION (abandoned):
Tried predicting total overtakes_made per driver-race using race-summary 
predictors: grid_position, pace_delta (driver's mean pace vs the mean pace of 
the driver who finished directly ahead), and pit_count as a tyre-strategy proxy. 
R²=0.114. grid_position and pit_count came out highly significant with sensible 
signs; pace_delta was essentially zero (p=0.83).

WHY THIS APPROACH IS FUNDAMENTALLY WRONG FOR THE QUESTION:
Overtakes are event-level phenomena. Each overtake happens at a specific lap, 
against a specific car, under specific conditions (tyre delta at that moment, 
gap to car ahead at that moment, DRS state, SC restart status, track section). 
Aggregating everything to race-level averages loses almost all the information 
that determines whether an overtake actually happens.

Specific failures of the race-level model:
  - "Pace delta vs car ahead" was implemented as "vs the driver who finished 
    directly ahead" -- but that's not the driver you actually battled during 
    overtake attempts throughout the race. Race-finish neighbors are similar by 
    definition (small pace deltas by structure), which is why this variable 
    came out completely non-significant.
  - Grid position captures a structural fact (started further back = more cars 
    to pass) but doesn't predict overtake SUCCESS at any specific moment.
  - Pit count is more likely picking up race chaos than tyre-strategy 
    advantage.

RESTARTING AT EVENT LEVEL:
Rebuilding the analysis as one row per potential overtake opportunity, with 
predictors captured at the moment of the attempt: gap to car ahead in seconds, 
tyre age delta at that moment, DRS/SC status at that moment, and race context. 
Outcome: did an overtake happen shortly after that moment.

This is a substantial rebuild of the data pipeline, but it's the only honest 
way to answer "what predicts overtakes" -- the question is inherently about 
moments, not race averages.

In [51]:
import time

opportunity_query = """
WITH lap_positions AS (
    SELECT l.session_key, l.driver_number, l.lap_number, l.date_start,
           (SELECT p.position
            FROM silver_position p
            WHERE p.session_key = l.session_key 
              AND p.driver_number = l.driver_number
              AND p.date <= l.date_start
            ORDER BY p.date DESC LIMIT 1) AS position_at_lap_start
    FROM silver_laps l
    JOIN silver_sessions s ON l.session_key = s.session_key AND s.session_name = 'Race'
    WHERE l.date_start IS NOT NULL AND l.lap_number >= 2
),
opportunities AS (
    SELECT lp.*,
           (SELECT p2.driver_number
            FROM silver_position p2
            WHERE p2.session_key = lp.session_key
              AND p2.position = lp.position_at_lap_start - 1
              AND p2.date <= lp.date_start
            ORDER BY p2.date DESC LIMIT 1) AS driver_ahead,
           (SELECT i.interval_seconds
            FROM silver_intervals i
            WHERE i.session_key = lp.session_key
              AND i.driver_number = lp.driver_number
              AND i.date <= lp.date_start
              AND i.interval_seconds IS NOT NULL
            ORDER BY i.date DESC LIMIT 1) AS gap_to_ahead
    FROM lap_positions lp
    WHERE lp.position_at_lap_start > 1
),
lap_end_dates AS (
    SELECT session_key, driver_number, lap_number, date_start
    FROM silver_laps
    WHERE date_start IS NOT NULL
)
SELECT o.*,
       (SELECT COUNT(*)
        FROM silver_overtakes ot
        JOIN lap_end_dates led ON led.session_key = ot.session_key 
                              AND led.driver_number = o.driver_number
                              AND led.lap_number = o.lap_number + 2
        WHERE ot.session_key = o.session_key
          AND ot.overtaking_driver_number = o.driver_number
          AND ot.overtaken_driver_number = o.driver_ahead
          AND ot.date > o.date_start
          AND ot.date <= led.date_start) AS overtook,
       (SELECT st.tyre_age_at_start + (o.lap_number - st.lap_start)
        FROM silver_stints st
        WHERE st.session_key = o.session_key
          AND st.driver_number = o.driver_number
          AND o.lap_number BETWEEN st.lap_start AND st.lap_end
        LIMIT 1) AS driver_tyre_age,
       (SELECT st.tyre_age_at_start + (o.lap_number - st.lap_start)
        FROM silver_stints st
        WHERE st.session_key = o.session_key
          AND st.driver_number = o.driver_ahead
          AND o.lap_number BETWEEN st.lap_start AND st.lap_end
        LIMIT 1) AS ahead_tyre_age
FROM opportunities o
WHERE o.gap_to_ahead < 2
"""

t0 = time.time()
df_opps = pd.read_sql(opportunity_query, conn)
print(f"Query took {time.time() - t0:.1f} seconds")
print(df_opps.shape)

Query took 100.2 seconds
(29597, 10)


In [57]:
df_opps['overtook'].value_counts()
df_opps[['gap_to_ahead','driver_tyre_age','ahead_tyre_age']].describe()
df_opps[['driver_tyre_age','ahead_tyre_age']].isna().sum()

driver_tyre_age    2911
ahead_tyre_age     2901
dtype: int64

In [58]:
df_opps['overtook_binary'] = (df_opps['overtook'] >= 1).astype(int)
df_opps['tyre_delta'] = df_opps['driver_tyre_age'] - df_opps['ahead_tyre_age']

df_opps_clean = df_opps.dropna(subset=['tyre_delta']).copy()

print(df_opps_clean.shape)
print(df_opps_clean['overtook_binary'].value_counts(normalize=True))
print(df_opps_clean['tyre_delta'].describe())

(26650, 12)
overtook_binary
0    0.833058
1    0.166942
Name: proportion, dtype: float64
count    26650.000000
mean        -0.934071
std          7.241680
min        -62.000000
25%         -1.000000
50%          0.000000
75%          1.000000
max         56.000000
Name: tyre_delta, dtype: float64


In [59]:
import statsmodels.formula.api as smf

model = smf.logit(
    'overtook_binary ~ gap_to_ahead + tyre_delta',
    data=df_opps_clean
).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.417058
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:        overtook_binary   No. Observations:                26650
Model:                          Logit   Df Residuals:                    26647
Method:                           MLE   Df Model:                            2
Date:                Mon, 20 Jul 2026   Pseudo R-squ.:                 0.07527
Time:                        11:42:20   Log-Likelihood:                -11115.
converged:                       True   LL-Null:                       -12019.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -0.5273      0.037    -14.215      0.000      -0.600      -0.455
gap_to_ahead    -1.2910

In [60]:
# grid_position needs to be pulled per race and merged

grid_query = """
SELECT g.driver_number, g.position AS grid_position, gs.meeting_key,
       (SELECT rs.session_key FROM silver_sessions rs 
        WHERE rs.meeting_key = gs.meeting_key AND rs.session_name = 'Race' LIMIT 1) AS session_key
FROM silver_starting_grid g
JOIN silver_sessions gs ON g.session_key = gs.session_key
WHERE gs.session_name = 'Qualifying'
"""
df_grid = pd.read_sql(grid_query, conn)

df_opps_clean = df_opps_clean.merge(
    df_grid[['session_key','driver_number','grid_position']],
    on=['session_key','driver_number'], how='left'
)

# check for merge gaps
print(df_opps_clean['grid_position'].isna().sum())
df_opps_clean = df_opps_clean.dropna(subset=['grid_position']).copy()
print(df_opps_clean.shape)

2463
(24187, 13)


In [61]:
model_ext = smf.logit(
    'overtook_binary ~ gap_to_ahead + tyre_delta + grid_position',
    data=df_opps_clean
).fit()
print(model_ext.summary())

Optimization terminated successfully.
         Current function value: 0.418395
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:        overtook_binary   No. Observations:                24187
Model:                          Logit   Df Residuals:                    24183
Method:                           MLE   Df Model:                            3
Date:                Mon, 20 Jul 2026   Pseudo R-squ.:                 0.08332
Time:                        11:52:05   Log-Likelihood:                -10120.
converged:                       True   LL-Null:                       -11040.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -0.3516      0.056     -6.304      0.000      -0.461      -0.242
gap_to_ahead     -1.

REBUILT AS EVENT-LEVEL LOGISTIC REGRESSION
Rebuilt the entire analysis at the opportunity level:
  - Unit of analysis: one row per (driver, lap) where the driver was within 
    2 seconds of the car ahead at the start of that lap. Filtered to Race 
    sessions only, lap >= 2 (excluding start-line chaos).
  - Outcome: overtook_binary = did an overtake by this driver against this 
    specific driver_ahead occur within the next 2 laps.
  - Predictors: gap_to_ahead (at that moment), tyre_delta (driver_tyre_age - 
    ahead_tyre_age at that moment), grid_position (race-level attribute added 
    to match the question's framing).
  - Built via correlated subqueries in SQLite, resolved to 26,650 valid 
    opportunities after dropping NaN tyre_delta (loss of ~10%, from stint 
    boundary edge cases and 2023 sparse-stint sessions). Further dropped to 
    24,187 for the grid_position version.

DATA VALIDATION DURING BUILD:
  - Verified on session 11253 (2026 Japanese GP) that positions evolved 
    plausibly lap-by-lap.
  - Verified driver_ahead identifications made sense against known race 
    sequence (Verstappen behind Leclerc for 10+ laps chasing).
  - Confirmed 16.5% overtake conversion rate matches F1 racing intuition 
    (most close-following moments don't convert to passes).

FINAL MODEL RESULTS (logistic regression, n=24,187):
  Pseudo R² = 0.083 (meaningful for real-world event prediction)
  All three predictors significant at p<0.001:
  
    gap_to_ahead:    coef=-1.319 -- dominant predictor. 
                     Closer gap -> much higher overtake probability. Every 
                     additional 1s of gap reduces log-odds by 1.32. Confirms 
                     the physics: at 0.5s you're threatening every corner; 
                     at 1.5s the car ahead can defend more easily.
    
    tyre_delta:      coef=-0.054
                     Fresher tyres (negative tyre_delta) -> higher overtake 
                     probability. Each 1-lap tyre advantage reduces log-odds 
                     by 0.05. Per-unit effect is smaller than gap, but spans 
                     a much larger range (up to 20+ lap deltas), so total 
                     influence is meaningful. Sign matches classic F1 "fresh 
                     tyres win" intuition.
    
    grid_position:   coef=-0.012
                     Negative coefficient came out OPPOSITE to intuition 
                     ("start further back = more attacking = more overtakes 
                     converted"). Actually says: starting further back 
                     corresponds to LOWER overtake conversion rate per 
                     opportunity. Small effect but significant (thanks to 
                     large sample). Likely explanation: grid_position 
                     proxies for unobserved quality -- better cars and 
                     better drivers qualify better AND convert overtakes 
                     more reliably. So the "attacking motivation from a bad 
                     grid" story is real but masked by stronger car/driver 
                     quality confounds.

TAKEAWAY:
Overtake conversion is driven primarily by moment-level context, not 
structural race features:
  1. Gap to the car ahead is the dominant predictor -- the closer, the more 
     likely the overtake converts.
  2. Fresh tyres relative to the car ahead genuinely help, with smaller 
     per-unit effect but meaningful range.
  3. Grid position has a small, subtle effect that likely proxies for 
     unobserved car/driver quality rather than any "attacking motivation" 
     mechanism.

The event-level model captures what the race-average model fundamentally 
could not: which specific opportunities convert and why. This is much closer 
to how overtakes actually happen in F1.

METHODOLOGICAL LESSON:
When a question asks about event-level phenomena (overtakes, incidents, 
critical moments), aggregating to race-level summaries is almost always the 
wrong approach even when it's tempting due to data simplicity. Race averages 
hide the moment-level variance that IS the signal. Building event-level 
analyses requires more work (correlated subqueries, careful timestamp 
lookups, condition-based filtering) but produces genuinely informative 
answers to the actual question, not misleading answers to a simpler 
substitute question.

- [X] Are the biggest position swings mostly explained by pit cycles and Safety Cars (as suspected from the Norris case), or is there a residual, unexplained portion once those are controlled for? (regression with pit-stop/Safety-Car dummy variables, examine residuals)

In [78]:
swing_query = """
WITH lap_positions AS (
    SELECT l.session_key, l.driver_number, l.lap_number, l.date_start,
           (SELECT p.position
            FROM silver_position p
            WHERE p.session_key = l.session_key 
              AND p.driver_number = l.driver_number
              AND p.date <= l.date_start
            ORDER BY p.date DESC LIMIT 1) AS position_at_lap_start
    FROM silver_laps l
    JOIN silver_sessions s ON l.session_key = s.session_key AND s.session_name = 'Race'
    WHERE l.date_start IS NOT NULL AND l.lap_number >= 2
)
SELECT lp1.session_key, lp1.driver_number, lp1.lap_number,
       lp1.position_at_lap_start AS pos_start,
       lp2.position_at_lap_start AS pos_end,
       lp2.position_at_lap_start - lp1.position_at_lap_start AS position_swing
FROM lap_positions lp1
JOIN lap_positions lp2 
  ON lp1.session_key = lp2.session_key 
 AND lp1.driver_number = lp2.driver_number
 AND lp2.lap_number = lp1.lap_number + 1
"""
df_swings = pd.read_sql(swing_query, conn)
df_swings.shape
df_swings['position_swing'].describe()

count    73807.000000
mean        -0.007696
std          0.918749
min        -13.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         15.000000
Name: position_swing, dtype: float64

In [79]:
pit_query = """
SELECT DISTINCT session_key, driver_number, lap_number, 1 AS pit_flag
FROM silver_pit
"""
df_pits = pd.read_sql(pit_query, conn)

sc_query = """
SELECT DISTINCT session_key, lap_number, 1 AS sc_flag
FROM silver_race_control
WHERE category = 'SafetyCar' AND lap_number IS NOT NULL
"""
df_sc = pd.read_sql(sc_query, conn)

# Merge onto swings
df_swings = df_swings.merge(df_pits, on=['session_key','driver_number','lap_number'], how='left')
df_swings = df_swings.merge(df_sc, on=['session_key','lap_number'], how='left')

df_swings['pit_flag'] = df_swings['pit_flag'].fillna(0).astype(int)
df_swings['sc_flag'] = df_swings['sc_flag'].fillna(0).astype(int)

df_swings.shape
df_swings[['pit_flag','sc_flag']].sum()
df_swings.groupby(['pit_flag','sc_flag'])['position_swing'].agg(['mean','std','count'])

mean       std  count
pit_flag sc_flag                           
0        0       -0.032210  0.860890  69822
         1       -0.088026  0.918693   1829
1        0        0.922713  1.835651   1902
         1        0.342520  1.857791    254

In [80]:
import statsmodels.formula.api as smf

model = smf.ols(
    'position_swing ~ pit_flag + sc_flag',
    data=df_swings
).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         position_swing   R-squared:                       0.027
Model:                            OLS   Adj. R-squared:                  0.027
Method:                 Least Squares   F-statistic:                     1021.
Date:                Mon, 20 Jul 2026   Prob (F-statistic):               0.00
Time:                        14:05:44   Log-Likelihood:                -97466.
No. Observations:               73807   AIC:                         1.949e+05
Df Residuals:                   73804   BIC:                         1.950e+05
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0307      0.003     -8.968      0.0

In [82]:
df_swings['predicted'] = model.predict(df_swings)
df_swings['residual'] = df_swings['position_swing'] - df_swings['predicted']

# Big residuals = swings unexplained by pit/SC
big_residuals = df_swings[df_swings['residual'].abs() >= 3].copy()
print(f"Rows with |residual| >= 3: {len(big_residuals)}")
big_residuals[['session_key','driver_number','lap_number','position_swing','pit_flag','sc_flag','residual']].head(20)

Rows with |residual| >= 3: 1366


,session_key,driver_number,lap_number,position_swing,pit_flag,sc_flag,residual
153,7779,20,9,4.0,0,0,4.030716
210,7779,24,12,6.0,0,0,6.030716
212,7779,27,12,3.0,0,0,3.030716
244,7779,18,14,6.0,0,0,6.030716
268,7779,10,15,5.0,0,0,5.030716
283,7779,55,16,5.0,0,0,5.030716
304,7779,16,17,3.0,0,0,3.030716
306,7779,31,17,5.0,0,0,5.030716
327,7779,23,18,4.0,0,1,4.145094
329,7779,21,18,5.0,0,1,5.145094


Driver 20, session 7779, lap 9
Actual position_swing: +4 (lost 4 places)
pit_flag = 0, sc_flag = 0 → model predicted the baseline: ~-0.031 (essentially "no change expected")
Residual = 4 - (-0.0307) = +4.03

CONTEXT ON POSITION SWINGS -- F1 Mechanics
Position swings during a race have three main mechanical causes:

1. PIT STOP DROP: A pit stop costs ~20-25 seconds of race time. A driver in 
   P3 pitting will drop to ~P8-9 as cars behind fly past on the main straight. 
   This is temporary -- as others pit later, the driver swings back up.

2. SAFETY CAR CYCLE: Under SC, cars slow ~40%, so a pit stop only costs 
   ~10-12s relative to the field vs. the usual 20+s. This can cause:
     - Positive swing: driver pits under SC while rivals already pitted 
       (they get a "free" stop and inherit positions)
     - Negative swing: driver pitted just before SC came out (they paid full 
       price while rivals now pit cheaply)

3. STAY-OUT GAMBLE: Late SC lets some cars stay out on old tyres while 
   leaders pit for fresh rubber. Backmarkers instantly swing up to top-5 
   positions. Reality: they can't hold those positions on old tyres once 
   racing restarts, so a large downward swing follows a few laps later.

Given these mechanics, we expect pit_flag and sc_flag to absorb most of the 
variance in position swings when regressed against them. Residuals -- laps 
with big swings that AREN'T explained by pit or SC context -- would 
represent "real racing" events: on-track battles, driver mistakes, 
mechanical incidents, weather transitions.

INTERPRETATION:
Given these mechanics: When drivers experience big position swings in a race moments where they gain or lose several places quickly, how much of that is explained by known, structural causes (making a pit stop, being caught out by a Safety Car), versus how much is 'unexplained' movement that isn't obviously tied to either?
The answer is the Residuals 


Coefficients :all sensible and as predicted:

- Intercept = -0.031: baseline swing when there's no pit and no SC. Essentially zero (drivers hold position on normal laps).
- pit_flag = +0.899 — pitting adds ~0.9 places lost on average. Matches your "Drop" mechanic.
- sc_flag = -0.114 — SC alone slightly reduces swings by ~0.1 places. Makes sense — SC discourages independent moves.
- All three p-values < 0.001. Statistically robust findings.

*But the headline is R² = 0.027 — only 2.7% of variance explained*

This is the surprising result. Despite pit_flag being a strong, real predictor with a huge coefficient (~0.9 places per pit stop), the two flags together explain only 2.7% of the total variance in position swings.

Why is R² so low despite pit_flag being so strong?

Because 94.6% of your rows have position_swing = 0 (no swing at all)

For the WHY (per event): when a big swing happens, pit is almost always the cause. Coefficient of +0.9 is huge and highly significant.
For the OVERALL variance: pit/SC explains only ~3% because most laps aren't swings at all. That "unexplained" 97% is mostly rows with swing=0 that the model correctly ignores anyway.



What these unexplained big swings actually represent:

- On-track battles that broke, extended fighting for position finally resolved one way, jumping multiple places at once.
- Race incidents, someone spun, went off-track, had a mechanical issue and dropped multiple places
- Weather transitions (some drivers adapted faster, causing multi-place shuffles).
- SC that happened but wasn't captured by our flag (remember your Notes item #10: race_control SC tags aren't exhaustive. Some SC laps could be missing from our flag.)
- VSC (Virtual Safety Car): different category from SafetyCar in race_control, and we didn't include it as a flag. Yet VSC has similar strategic effects.

Total unexplained big swings: 1,366 rows : that's about 1.9% of your 73,807 total, but it's arguably the more analytically interesting portion. The 97% of "no swing" rows are the boring "everyone holding position" cases; these 1,366 are where real racing happened.